In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import  load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")



## Summerization Middleware

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

from langgraph.checkpoint.memory import InMemorySaver

from langchain_core.messages import HumanMessage, SystemMessage


agent = create_agent(
    model = "groq:llama-3.3-70b-versatile",
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = "groq:llama-3.3-70b-versatile",
            trigger=("messages",10),
            keep = ("messages",4)
        )
    ]
)

In [9]:
config = {"configurable":{"thread_id":"test-1"}}

In [11]:
questions = [
    "what is 2+2",
    "what is 10*5",
    "what is 100/4",
    "what is 15-7",
    "what is 3*3",
    "what is 4*4",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"message:{response}")
    print(f"message:{len(response["messages"])}")


message:{'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='c5dec964-b8a1-4516-83f9-1c7fd4b1640d'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 41, 'total_tokens': 50, 'completion_time': 0.021132594, 'completion_tokens_details': None, 'prompt_time': 0.001959194, 'prompt_tokens_details': None, 'queue_time': 0.091698159, 'total_time': 0.023091788}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6a31-2d0f-7112-921f-1c623c0b5d9d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 9, 'total_tokens': 50})]}
message:2
message:{'messages': [HumanMessage(content='what is 2+2', additional_kwargs={}, response_metadata={}, id='c5dec964-b8a1-4516-83f9-1c7fd4b1640d'), AIMessage(cont

## Token size

In [14]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

# Create tool
@tool
def search_hotels(city: str) -> str:
    """Search hotels in a city."""

    return f"""
Hotels in {city}:

1. Grand Hotel
   - 5 star
   - $350/night
   - spa, pool, gym

2. City Inn
   - 4 star
   - $180/night
   - business center

3. Budget Stay
   - 3 star
   - $75/night
   - free wifi
"""


# Create agent
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",

    # Memory
    checkpointer=InMemorySaver(),

    # Tools must be a list
    tools=[search_hotels],

    # Middleware
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",

            # Summarize after 550 tokens
            max_tokens_before_summary=550,

            # Keep latest messages
            messages_to_keep=5
        ),
    ]
)


# Thread configuration
config = {
    "configurable": {
        "thread_id": "test-1"
    }
}


# Approx token counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars



C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14372\2869649997.py:44: DeprecationWarning: max_tokens_before_summary is deprecated. Use trigger=('tokens', value) instead.
  SummarizationMiddleware(
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_14372\2869649997.py:44: DeprecationWarning: messages_to_keep is deprecated. Use keep=('messages', value) instead.
  SummarizationMiddleware(


In [15]:

# Test cities
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]


# Run test
for city in cities:

    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=f"Find hotels in {city}")
            ]
        },
        config=config
    )

    tokens = count_tokens(response["messages"])

    print(f"\n{city}: ~{tokens} characters, {len(response['messages'])} messages\n")

    print("============== RESPONSE ==============\n")

    for msg in response["messages"]:
        print(msg)
        print()

    print("======================================")


Paris: ~1571 characters, 8 messages

============== RESPONSE ==============

content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nFind hotels in Paris.\n\n## SUMMARY\nThe user requested a list of hotels in Paris. Two separate searches were conducted, both returning the same list of hotels: Grand Hotel (5-star, $350/night, with spa, pool, and gym), City Inn (4-star, $180/night, with business center), and Budget Stay (3-star, $75/night, with free wifi).\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe user needs to select a hotel from the provided list or request additional information about one of the hotels to make a decision.' additional_kwargs={'lc_source': 'summarization'} response_metadata={} id='e0d8d272-0335-4543-ada4-5ab3daf9e8f6'

content='' additional_kwargs={'tool_calls': [{'id': 'wmhvb5ezk', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tok

## Fraction

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~117 tokens (0.0914%), 6 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='13a75faa-44ea-4dba-b400-8a52eabed7ab'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'njs7rnftc', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 210, 'total_tokens': 225, 'completion_time': 0.035820028, 'completion_tokens_details': None, 'prompt_time': 0.010454256, 'prompt_tokens_details': None, 'queue_time': 0.091495331, 'total_time': 0.046274284}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6a58-c914-7930-a3f0-3ee975f19f51-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'njs7rnftc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metad

### Human In the Loop MiddleWare

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"



In [14]:
agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)


In [15]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [16]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='1b1cc954-b013-420e-8d4d-6a66e8c09fe3'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8kdgedk2r', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 309, 'total_tokens': 341, 'completion_time': 0.049893122, 'completion_tokens_details': None, 'prompt_time': 0.015526324, 'prompt_tokens_details': None, 'queue_time': 0.048994303, 'total_time': 0.065419446}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6a6f-bde7-7221-9a45-6fb83f1b8b78-0', tool_calls=[{'name': 'send_email_tool', 'args': {'b

In [17]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Email sent to john@test.com with subject 'Hello' and body 'How are you?'
